In [1]:
# Python ≥3.5 is required
import sys

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

if IS_COLAB:
    %pip install -q -U tensorflow-addons
    %pip install -q -U transformers

# Scikit-Learn ≥0.20 is required
import sklearn

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras

if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. LSTMs and CNNs can be very slow without a GPU.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware accelerator.")
    if IS_KAGGLE:
        print("Go to Settings > Accelerator and select GPU.")

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 103.8 MB/s eta 0:00:00


In [2]:
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response=requests.get(url)
with open('tiny_shakespeare.txt', 'w', encoding='utf-8') as f:
    f.write(response.text)
with open('tiny_shakespeare.txt') as f:
    shakespeare_text = f.read()

In [3]:
print(shakespeare_text[:148])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?



In [4]:
tokenizer=keras.preprocessing.text.Tokenizer(char_level=True)
tokenizer.fit_on_texts(shakespeare_text)

In [5]:
tokenizer.texts_to_sequences(["First"])

[[20, 6, 9, 8, 3]]

In [6]:
tokenizer.sequences_to_texts([[20, 6, 9, 8, 3]])

['f i r s t']

In [7]:
max_id=len(tokenizer.word_index)
dataset_size=tokenizer.document_count

In [8]:
[encoded]=np.array(tokenizer.texts_to_sequences([shakespeare_text]))-1
train_size=dataset_size*90//100
dataset=tf.data.Dataset.from_tensor_slices(encoded[:train_size])

In [9]:
n_steps=100
window_length=n_steps+1
dataset=dataset.window(window_length,shift=1,drop_remainder=True)

In [10]:
dataset=dataset.flat_map(lambda window:window.batch(window_length))

In [11]:
np.random.seed(42)
tf.random.set_seed(42)


In [12]:
batch_size=32
dataset=dataset.shuffle(10000).batch(batch_size)
dataset=dataset.map(lambda windows:(windows[:,:-1],windows[:,1:]))

In [13]:
dataset=dataset.map(
    lambda X_batch,Y_batch:(tf.one_hot(X_batch,depth=max_id),Y_batch)
)

In [14]:
dataset=dataset.prefetch(1)

In [15]:
for X_batch,Y_batch in dataset.take(1):
    print(X_batch.shape,Y_batch.shape)

(32, 100, 39) (32, 100)


In [ ]:
model = keras.models.Sequential([
    keras.layers.GRU(128, return_sequences=True, input_shape=[None, max_id],
                     #dropout=0.2, recurrent_dropout=0.2),
                     dropout=0.2),
    keras.layers.GRU(128, return_sequences=True,
                     #dropout=0.2, recurrent_dropout=0.2),
                     dropout=0.2),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,
                                                    activation="softmax"))
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
history = model.fit(dataset, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 2159s 69ms/step - loss: 1.5978
Epoch 2/10


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31368/31368 ━━━━━━━━━━━━━━━━━━━━ 2119s 67ms/step - loss: 1.5155
Epoch 3/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 2137s 68ms/step - loss: 1.4961
Epoch 4/10
31368/31368 ━━━━━━━━━━━━━━━━━━━━ 2129s 68ms/step - loss: 1.4850
Epoch 5/10
23101/31368 ━━━━━━━━━━━━━━━━━━━━ 9:16 67ms/step - loss: 1.4810

KeyboardInterrupt: 

I just run 5 epochs you can run more if you want it will lead to better result but you can get much better results in stateful rnn .

In [39]:
def preprocess(texts):
    X = np.array(tokenizer.texts_to_sequences(texts)) - 1
    return tf.one_hot(X, max_id)

In [40]:
X_new = preprocess(["How are yo"])
Y_pred = np.argmax(model(X_new), axis=-1)
tokenizer.sequences_to_texts(Y_pred + 1)[0][-1]

'u'

In [ ]:
tf.random.set_seed(42)

tf.random.categorical([[np.log(0.5), np.log(0.4), np.log(0.1)]], num_samples=40).numpy()

array([[0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0,
        2, 0, 0, 1, 1, 1, 0, 0, 1, 2, 0, 0, 1, 1, 0, 0, 0, 0]])

In [41]:
def next_char(text, temperature=1):
    X_new = preprocess([text])
    y_proba = model(X_new)[0, -1:, :]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1) + 1
    return tokenizer.sequences_to_texts(char_id.numpy())[0]

In [42]:
tf.random.set_seed(42)

next_char("How are yo", temperature=1)

'u'

In [43]:
def complete_text(text, n_chars=100, temperature=1):
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text

In [ ]:
tf.random.set_seed(42)

print(complete_text("w", temperature=0.1))

with him to the prisoner of the queen
and so please you to the queen, and so please your gracious lord,
i come to me a man that she shall be perform.

paulina:
no, my lord,
i come to the prince and the thought of the prisoner,
that she shall be perform, the queen and many a man that she shall be perform.

leontes:
what is the queen, and so please your gracious lord,
i come to me as your prisoner shall not see
the queen and so please your gracious lord,
i come to me a man that she shall be perform


In [18]:
tf.random.set_seed(42)


In [19]:
dataset=tf.data.Dataset.from_tensor_slices(encoded[:train_size])
dataset=dataset.window(window_length,shift=n_steps,drop_remainder=True)
dataset=dataset.flat_map(lambda window:window.batch(window_length))
dataset=dataset.batch(1)
dataset=dataset.map(lambda windows:(windows[:,:-1],windows[:,1:]))
dataset=dataset.map(
    lambda X_batch,Y_batch:(tf.one_hot(X_batch,depth=max_id),Y_batch)
)
dataset=dataset.prefetch(1)

In [20]:
batch_size=32
encoded_parts=np.array_split(encoded[:train_size],batch_size)
datasets=[]
for encoded_part in encoded_parts:
  dataset=tf.data.Dataset.from_tensor_slices(encoded_part)
  dataset=dataset.window(window_length,shift=n_steps,drop_remainder=True)
  dataset=dataset.flat_map(lambda window:window.batch(window_length))
  datasets.append(dataset)
dataset = tf.data.Dataset.zip(tuple(datasets)).map(lambda *windows: tf.stack(windows))
dataset = dataset.map(lambda windows: (windows[:, :-1], windows[:, 1:]))
dataset = dataset.map(
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=max_id), Y_batch))
dataset = dataset.prefetch(1)

In [24]:
model = keras.Sequential([
    keras.Input(batch_shape=(batch_size, None, max_id)),
    keras.layers.GRU(
        128,
        return_sequences=True,
        stateful=True,
        dropout=0.2
    ),
    keras.layers.GRU(
        128,
        return_sequences=True,
        stateful=True,
        dropout=0.2
    ),
    keras.layers.TimeDistributed(
        keras.layers.Dense(max_id, activation="softmax")
    )
])

In [31]:
class ResetStateCallback(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        for layer in self.model.layers:
            if hasattr(layer, "reset_states"):
                layer.reset_states()

In [32]:
model.compile(loss="sparse_categorical_crossentropy",optimizer="adam")
history=model.fit(dataset,epochs=50,callbacks=[ResetStateCallback()])

Epoch 1/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 85ms/step - loss: 2.6224
Epoch 2/50


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - loss: 2.2405
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 82ms/step - loss: 2.1057
Epoch 4/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 82ms/step - loss: 2.0278
Epoch 5/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 81ms/step - loss: 1.9754
Epoch 6/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 81ms/step - loss: 1.9374
Epoch 7/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 84ms/step - loss: 1.9074
Epoch 8/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - loss: 1.8861
Epoch 9/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 82ms/step - loss: 1.8661
Epoch 10/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - loss: 1.8504
Epoch 11/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - loss: 1.8359
Epoch 12/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 84ms/step - loss: 1.8267
Epoch 13/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - loss: 1.8174
Epoch 14/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 86ms/step - loss: 1.8055
Epoch 15/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 83ms/step - loss: 1.7976
Epoch 16/50
313/313 ━━━━━━━━━━━━━━━━━━━

In [33]:
stateless_model=keras.models.Sequential([
    keras.layers.GRU(128,return_sequences=True,input_shape=[None,max_id]),
    keras.layers.GRU(128,return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,activation="softmax"))

])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [37]:
stateless_model.build(tf.TensorShape([None,None,max_id]))

In [38]:
stateless_model.set_weights(model.get_weights())
model=stateless_model

In [47]:
tf.random.set_seed(42)
print(complete_text("t",temperature=0.1))

ther the sent
that i will not so the country to the good
and the country should be the country should
